# Groundhog — a quantitative teardown 🔬
### The same-month vs other-month control · the Lo t-stat · no decay · the cost sweep

![Signal: Real](https://img.shields.io/badge/Signal-Real-2ea44f?style=flat-square)
![Tradability: Fragile](https://img.shields.io/badge/Tradability-Fragile-dab617?style=flat-square)
![Genuinely same-month seasonal?: Confirmed](https://img.shields.io/badge/Genuinely_same--month_seasonal%3F-Confirmed-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb). We confirm return seasonality is real and same-month-specific, then ask what survives turnover.

> ⚠️ **Not investment advice.** 398 *current* S&P 500 names with ≥20y history (Yahoo), 2000–2026 — a **twice-survivor panel** (current membership × long-history filter; `fetch_panel` requires an explicit `allow_survivorship_bias=True`). Magnitudes are upper bounds; the Signal call leans on the bias-sharing control and on Heston-Sadka's CRSP evidence. Sources in [`docs/references.md`](../docs/references.md).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))  # repo root (quantlab/)
sys.path.insert(0, os.path.abspath(".."))        # study package (groundhog/)
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.5); plt.rcParams["axes.grid"] = True
import numpy as np, pandas as pd
from groundhog import data, strategy as st
# Opt-in, stated: current S&P 500 members + a >=20y-history filter = a twice-survivor panel.
# Magnitudes below are upper bounds; the same-month-vs-other-month control shares the bias.
ret = data.fetch_panel(allow_survivorship_bias=True)   # cache-first; built by examples/verify.py --fetch
same = st.seasonal_hedge(ret, same_month=True)
ctrl = st.seasonal_hedge(ret, same_month=False)


C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\48-groundhog\groundhog\strategy.py:45: RuntimeWarning: Mean of empty slice
  pred = np.nanmean(R[rows, :], axis=0)


C:\Users\loutr\Dropbox\Perso\GitHub\Open-Alpha-Lab\studies\48-groundhog\groundhog\strategy.py:45: RuntimeWarning: Mean of empty slice
  pred = np.nanmean(R[rows, :], axis=0)


## Verdict, up front

| Axis | Stamp | Why |
|---|---|---|
| Signal | **Real** | +7.3%/yr, Sharpe 0.81, Lo t 4.1, no decay — survivor panel, so magnitude inflated; existence backed by the bias-sharing control + CRSP |
| Tradability | **Fragile** | one-way break-even ≈19 bp; net Sharpe 0.38 at 10 bp, 0.33–0.36 after borrow |
| Same-month seasonal? | **Confirmed** | other-month control earns −3.3%/yr |

> 💡 *In plain words:* a genuinely strange effect that passes every guard we can run — measured on a universe of winners, so we trust the *that*, not the *how much*.

## 1 · The claim, steelmanned

- **H₁:** the same-month long-short earns a significant positive return.
- **H₂:** it's specific to the same month (the other-month control fails).
- **H₃:** it survives realistic costs.

**Known bias, named up front:** the panel is current S&P 500 membership *and* a ≥20y-history filter — conditioned on the future twice. H₂'s control shares the bias (it differences out of the comparison); H₁'s *magnitude* does not get that protection.

## 2 · So what? — what rides on each

If H₁/H₂ hold, calendar seasonality is a real, decorrelated alpha and a puzzle for efficient markets. H₃ decides whether it's investable or merely true.

## 3 · How we'd know — the protocol

Score by same-month history (trailing **up to** 20 years, ≥5 same-month obs — the early sample ranks on 5–12 observations) → long-short → Lo t-stat → the other-month control → decade split → one-way cost sweep, break-even, borrow.

## 4 · The teardown

### 4.1 The effect and the control

In [2]:
import pandas as pd
display(pd.DataFrame({'same-month':st.stats(same),'control (other months)':st.stats(ctrl)}).T[['mean_ann','sharpe','tstat','hit_rate','n']].round(3))

,mean_ann,sharpe,tstat,hit_rate,n
same-month,0.073,0.805,4.090,0.638,318.0
control (other months),-0.033,-0.212,-1.088,0.509,318.0


> 💡 *In plain words:* +7.3%/yr at t 4.1 for the same month; the control is negative. **H₁ and H₂ hold** — with the caveat split where it belongs: both arms run on the same survivor panel, so the *contrast* (H₂) is bias-robust while the *level* (H₁'s +7.3%/yr) is an upper bound. The vendor lists this strategy at Sharpe **0.340** on a broader, honest universe — our 0.81 is what a large-cap twice-survivor panel shows, not what you'd have earned; the truth likely sits nearer 0.34.

### 4.2 No decay

In [3]:
for lab,sl in [('2000-2012',same.loc[:'2012']),('2013-on',same.loc['2013':])]:
    print(f'{lab}: Sharpe {st.stats(sl)["sharpe"]:+.2f}, mean {st.stats(sl)["mean_ann"]:+.2%}/yr')

2000-2012: Sharpe +0.81, mean +8.32%/yr
2013-on: Sharpe +0.81, mean +6.37%/yr


> 💡 *In plain words:* identical across halves — the hallmark of a real effect, not a mined one. (Sample starts 2000-01 after the 60-month warmup on a 1995 panel.)

### 4.3 The cost sweep — turnover counted one-way

In [4]:
rows={'gross':st.stats(same)['sharpe']}
for c in (5,10,20): rows[f'{c}bp']=st.stats(st.net_of_cost(same,c))['sharpe']
import pandas as pd; display(pd.Series(rows, name='net Sharpe (one-way bp)').round(3))
print(f'break-even ≈ {st.breakeven_cost_bps(same):.0f} bp one-way (3.2x NAV traded/mo)')
for b in (25,50):
    print(f'10 bp + {b} bp/yr short borrow: net Sharpe {st.stats(st.net_of_borrow(st.net_of_cost(same,10),b))["sharpe"]:+.2f}')

gross    0.805
5bp      0.594
10bp     0.383
20bp    -0.039
Name: net Sharpe (one-way bp), dtype: float64

break-even ≈ 19 bp one-way (3.2x NAV traded/mo)
10 bp + 25 bp/yr short borrow: net Sharpe +0.36
10 bp + 50 bp/yr short borrow: net Sharpe +0.33


> 💡 *In plain words:* replacing ~80% of a two-sided book is ~3.2× NAV of one-way trades a month, so break-even is **~19 bp one-way** (an earlier draft charged round-trip and said 38). Net Sharpe is 0.38 at 10 bp and the ~80-name short book's borrow (25–50 bp/yr) trims that to 0.33–0.36. Alive at a few bp; dead by 20. **H₃ barely holds → Fragile.**

## 5 · The verdict

H₂ holds outright; H₁ holds on existence (magnitude inflated by the survivor panel); H₃ barely → Signal `REAL` *with the magnitude caveat*, seasonality `CONFIRMED`, Tradability `FRAGILE`.

## 6 · Could you trade it?

Only as a tightly-costed market-neutral overlay — at institutional one-way costs (≲5 bp) it keeps a net Sharpe ~0.6 *on this flattered panel*; at retail costs it's gone, and short borrow, monthly 3.2× turnover and capacity all bite before that. It's the rare *real* one here; the honest caveats are the panel and the implementation.

## 7 · Going further

Forks: (a) a point-in-time / small-cap universe (the effect is stronger but less tradable); (b) the seasonal *factor* (Keloharju-Linnainmaa-Nyberg 2016) vs stock-level; (c) a long-only tilt to drop the short book. Backlog: [`docs/pwb_strategies_inventory.md`](../../../docs/pwb_strategies_inventory.md).